In [1]:
!pip install pandas pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 37.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import pymorphy3
from collections import defaultdict
import re
import os

def lemmatize_text(text, morph):
    if pd.isna(text) or not isinstance(text, str):
        return []

    words = re.findall(r'\b[а-яё]+\b', text.lower(), flags=re.IGNORECASE)

    lemmatized = []
    for word in words:
        try:
            parsed = morph.parse(word)[0]
            lemmatized.append(parsed.normal_form)
        except:
            lemmatized.append(word)

    return lemmatized

def read_file_safe(file_path):
    try:
        file_ext = os.path.splitext(file_path)[1].lower()

        if file_ext == '.tsv':
            separator = '\t'
            file_type = 'TSV'
        elif file_ext == '.csv':
            separator = ','
            file_type = 'CSV'
        else:
            with open(file_path, 'r', encoding='utf-8') as f:
                first_line = f.readline()
                if '\t' in first_line:
                    separator = '\t'
                    file_type = f'{file_ext} (auto-detected as TSV)'
                elif ',' in first_line:
                    separator = ','
                    file_type = f'{file_ext} (auto-detected as CSV)'
                else:
                    raise ValueError(f"Cannot auto-detect separator for {file_path}")

        encodings = ['utf-8', 'cp1251', 'koi8-r', 'utf-8-sig']
        for encoding in encodings:
            try:
                df = pd.read_csv(file_path, encoding=encoding, sep=separator)
                print(f"Successfully read {file_path} as {file_type} with {encoding} encoding")
                print(f"Found {len(df)} rows and {len(df.columns)} columns")
                return df
            except UnicodeDecodeError:
                continue
            except Exception as e:
                print(f"Error reading with {encoding}: {e}")
                continue

        raise ValueError(f"Could not read {file_path} with any supported encoding")

    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

def count_words_by_level(source_df, base_df):
    try:
        morph = pymorphy3.MorphAnalyzer()
        print("Successfully initialized pymorphy3")
    except Exception as e:
        print(f"Error initializing pymorphy3: {e}")
        raise

    base_words_by_level = defaultdict(set)
    for _, row in base_df.iterrows():
        level = row['level'][0]
        if pd.notna(row['base']):
            base_word = str(row['base']).lower().strip()
            if base_word:
                base_words_by_level[level].add(base_word)

    total_counts = defaultdict(int)
    word_counts = defaultdict(lambda: defaultdict(int))
    sentences_processed = 0
    total_words_found = 0

    for idx, row in source_df.iterrows():
        # source_text = row['source']
        source_text = row['result']

        if pd.isna(source_text) or not isinstance(source_text, str):
            continue

        lemmatized_words = lemmatize_text(source_text, morph)

        words_found_in_sentence = set()

        for level, base_words in base_words_by_level.items():
            for word in lemmatized_words:
                if word in base_words and word not in words_found_in_sentence:
                    total_counts[level] += 1
                    word_counts[level][word] += 1
                    words_found_in_sentence.add(word)
                    total_words_found += 1

        if lemmatized_words:
            sentences_processed += 1

    results = []
    for level in sorted(base_words_by_level.keys()):
        results.append({
            'level': level,
            'total_occurrences': total_counts.get(level, 0),
            'unique_words_found': len(word_counts.get(level, {})),
            'total_base_words': len(base_words_by_level.get(level, set()))
        })

    if results:
        results_df = pd.DataFrame(results)
    else:
        results_df = pd.DataFrame(columns=['level', 'total_occurrences', 'unique_words_found', 'total_base_words'])

    print(f"Processed {sentences_processed} sentences")
    print(f"Found {total_words_found} total word occurrences")

    return results_df

def main(source_file, base_file):

    try:
        print(f"\nReading source file: {source_file}")
        source_df = read_file_safe(source_file)

        print(f"\nReading base file: {base_file}")
        base_df = read_file_safe(base_file)

        if source_df is None or base_df is None:
            print("Failed to read input files")
            return

        source_col = source_df['result']
        base_col = base_df['base']
        level_col_base = base_df['level']

        print("\n" + "="*50)
        print("LEVEL DISTRIBUTION")
        print("="*50)

        print("\n" + "="*50)
        print("PROCESSING...")
        print("="*50)
        results_df = count_words_by_level(source_df, base_df)

        print("\n" + "="*50)
        print("RESULTS SUMMARY BY LEVEL")
        print("="*50)
        if not results_df.empty:
            print(results_df.to_string(index=False))
        else:
            print("No results found")

        if not results_df.empty:
            results_df.to_csv('level_counts_summary.csv', index=False, encoding='utf-8-sig')
            print(f"\n✓ Summary saved to 'level_counts_summary.csv'")


        print("\n" + "="*50)
        print("PROCESSING COMPLETE")
        print("="*50)

    except FileNotFoundError as e:
        print(f"\nError: File not found - {e}")
        print("\nPlease update the file paths in the main() function:")
        print("  source_file = 'path/to/your/source_file.tsv'  # or .csv")
        print("  base_file = 'path/to/your/base_file.csv'      # or .tsv")
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# изначальный датасет eval

if __name__ == "__main__":
    source_file = '/content/eval_source_target_sentences.csv'
    base_file = '/content/new_vocabulary.csv'
    main(source_file, base_file)

Successfully read /content/eval_source_target_sentences.csv with utf-8 encoding
Successfully read /content/new_vocabulary.csv with utf-8 encoding

Level distribution in source file:
level
b1       1950
b1_c1     392
b2        175
a2_b1      78
a2         77
b1_b2      13
Name: count, dtype: int64

Level distribution in base file:
level
B2    3422
B1    1562
C1    1393
A2    1056
A1     932
C2     582
Name: count, dtype: int64
Successfully initialized pymorphy3
Processed 2684 sentences
Found 33535 total word occurrences

SUMMARY BY LEVEL
level  total_occurrences  unique_words_found  total_base_words
    A              26243                1133              1988
    B               5735                1663              4984
    C               1557                 512              1975

Summary saved to 'level_counts_summary.csv'
Detailed results saved to 'level_counts_detailed.csv'


In [ ]:
# non-controlled baseline
if __name__ == "__main__":
    source_file = '/content/eval_result_noncontr_RUBERT2.сsv'
    base_file = '/content/new_vocabulary.csv'
    main(source_file, base_file)


Reading source file: /content/eval_result_noncontr_RUBERT2.сsv
Successfully read /content/eval_result_noncontr_RUBERT2.сsv as .сsv (auto-detected as TSV) with utf-8 encoding
Found 2685 rows and 2 columns

Reading base file: /content/new_vocabulary.csv
Successfully read /content/new_vocabulary.csv as CSV with utf-8 encoding
Found 8947 rows and 3 columns

LEVEL DISTRIBUTION

PROCESSING...
Successfully initialized pymorphy3
Processed 2683 sentences
Found 21898 total word occurrences

RESULTS SUMMARY BY LEVEL
level  total_occurrences  unique_words_found  total_base_words
    A              17267                1044              1988
    B               3652                1330              4984
    C                979                 406              1975

✓ Summary saved to 'level_counts_summary.csv'

PROCESSING COMPLETE


In [ ]:
# controlled baseline, level = A
if __name__ == "__main__":
    source_file = '/content/eval_result_contr_lvlA.tsv'
    base_file = '/content/new_vocabulary.csv'
    main(source_file, base_file)


Reading source file: /content/eval_result_contr_lvlA.tsv
Successfully read /content/eval_result_contr_lvlA.tsv as TSV with utf-8 encoding
Found 2685 rows and 3 columns

Reading base file: /content/new_vocabulary.csv
Successfully read /content/new_vocabulary.csv as CSV with utf-8 encoding
Found 8947 rows and 3 columns

LEVEL DISTRIBUTION

PROCESSING...
Successfully initialized pymorphy3
Processed 2682 sentences
Found 23537 total word occurrences

RESULTS SUMMARY BY LEVEL
level  total_occurrences  unique_words_found  total_base_words
    A              19602                1052              1988
    B               3300                1222              4984
    C                635                 270              1975

✓ Summary saved to 'level_counts_summary.csv'

PROCESSING COMPLETE


In [ ]:
# controlled baseline, level = B
if __name__ == "__main__":
    source_file = '/content/eval_result_contr_lvlB.tsv'
    base_file = '/content/new_vocabulary.csv'
    main(source_file, base_file)


Reading source file: /content/eval_result_contr_lvlB.tsv
Successfully read /content/eval_result_contr_lvlB.tsv as TSV with utf-8 encoding
Found 2685 rows and 3 columns

Reading base file: /content/new_vocabulary.csv
Successfully read /content/new_vocabulary.csv as CSV with utf-8 encoding
Found 8947 rows and 3 columns

LEVEL DISTRIBUTION

PROCESSING...
Successfully initialized pymorphy3
Processed 2684 sentences
Found 26930 total word occurrences

RESULTS SUMMARY BY LEVEL
level  total_occurrences  unique_words_found  total_base_words
    A              21565                1083              1988
    B               4536                1470              4984
    C                829                 352              1975

✓ Summary saved to 'level_counts_summary.csv'

PROCESSING COMPLETE


In [3]:
# controlled baseline, level = C
if __name__ == "__main__":
    source_file = '/content/eval_result_contr_lvlC.tsv'
    base_file = '/content/new_vocabulary.csv'
    main(source_file, base_file)


Reading source file: /content/eval_result_contr_lvlC.tsv
Successfully read /content/eval_result_contr_lvlC.tsv as TSV with utf-8 encoding
Found 2685 rows and 3 columns

Reading base file: /content/new_vocabulary.csv
Successfully read /content/new_vocabulary.csv as CSV with utf-8 encoding
Found 8947 rows and 3 columns

LEVEL DISTRIBUTION

PROCESSING...
Successfully initialized pymorphy3
Processed 2680 sentences
Found 34244 total word occurrences

RESULTS SUMMARY BY LEVEL
level  total_occurrences  unique_words_found  total_base_words
    A              26572                1087              1988
    B               6621                1496              4984
    C               1051                 382              1975

✓ Summary saved to 'level_counts_summary.csv'

PROCESSING COMPLETE
